**Cell 1**

# Train TinyLlama HelpSteer2 RS-PPO Adapters (ArmoRM)

Trains five independent TinyLlama LoRA specialists with PPO, one per HelpSteer2 attribute, using ArmoRM heads as the reward — the Rewarded-Soups recipe. This notebook is a thin driver: all training logic lives in the already-tested `scripts/train_rs_ppo.py` (RS Table-1 defaults, the verified ArmoRM scorer with its golden-sample and batching checks, Equal-N, plateau detection). We do not reimplement any of that here, and the RS-faithful hyperparameters inside the script must not be tuned.

**Rewarded-Soups fidelity.** RS's procedure is: run PPO once per reward, holding architecture, hyperparameters and RL algorithm identical across runs — *only the reward varies* — then interpolate the weights and slide λ to trace a front (Ramé et al., NeurIPS 2023). This notebook mirrors that exactly: five calls that differ only in `--axis`, with identical LoRA config, PPO hyperparameters, seeds, prompt set and step count (Equal-N). The two deliberate, thesis-critical divergences from RS are (1) reward = ArmoRM heads for training **and** evaluation → circular (RS uses independent reward models per task), and (2) θ_SFT = base TinyLlama via shortcut (RS uses LLaMA-7b + Alpaca-SFT). Both are documented design choices, not recipe drift; changing them (e.g. native reward models) is a recipe change that goes through Lingxiao.

**Reward-model precision (fixes the golden-sample crash).** ArmoRM is loaded in **bf16, not 4-bit**. 4-bit NF4 corrupts ArmoRM's regression heads — most on the quality heads (helpfulness 2.78→1.77, correctness 2.86→1.91) — enough to fail the golden-sample anchor (`max_abs 1.02 > atol 0.35`). In bf16 the anchor passes (`max_abs 0.06`). Every training call below therefore passes `--armorm_load_in_4bit false`. **Consistency requirement:** the later evaluation notebook must load ArmoRM in the *same* bf16 so that train-reward == eval-reward holds (the whole point of the acknowledged circular setup).

PPO reward and later evaluation both use ArmoRM, so this run is **deliberately circular**: RQ2 is retired; only the upper-bound / geometry / linearity readings are valid.

Scope of this notebook: produce the five adapters and zip them. Computing R is done later in NB05.

**Cell 2**

## 1. Clone or update the repository

In [ ]:
# Cell 3
%cd /content
import os, shutil
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

**Cell 4**

## 2. Check the GPU

An A100-40GB is required: the TinyLlama policy and the ArmoRM reward model load together during PPO. In bf16 ArmoRM alone is ~16 GB (vs ~6 GB in 4-bit), so the co-residence margin is tighter than before — the smoke test in Cell 17 verifies it fits before the real runs.

In [ ]:
# Cell 5
!nvidia-smi

**Cell 6**

## 3. Install dependencies

Same pinned set as the SFT notebook, plus `trl` for the PPO loop. ArmoRM declares Transformers 4.40.0 and its custom model code relies on that version's internal Llama API, so Transformers, PEFT, and Accelerate stay pinned.

In [ ]:
# Cell 7
!pip uninstall -y torchao
!pip install -q -U "pandas==2.2.2" "numpy<2.1" "protobuf>=5.29.1,<6.0.0" "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" "trl==0.8.6" bitsandbytes datasets pyyaml safetensors

**Cell 8**

Restart the runtime once after installation so Python forgets previously imported Transformers or PyTorch modules, then rerun the repository cell (Cell 3) and continue.

**Cell 9**

## 4. Settings

These are passed to `train_rs_ppo.py` as CLI overrides. Everything else (LoRA rank/alpha/dropout, lr, KL, output length 16–32, ppo_epochs) is fixed at RS Table-1 defaults inside the script and must not be tuned. `total_ppo_steps`, `n_prompts`, `prompt_seed`, and `train_seed` are identical across all five axes (Equal-N) — do not vary them per axis.

In [ ]:
# Cell 10
BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ARMORM_MODEL = "RLHFlow/ArmoRM-Llama3-8B-v0.1"
OUT_DIR      = "results/rs_ppo_armorm_circular/rs_runs"

TOTAL_PPO_STEPS = 200      # short horizon; identical for every axis
BATCH_SIZE      = 64       # RS uses 128; halved for Colab VRAM (use 128 on A100-80GB)
N_PROMPTS       = 2005

print(f"base   = {BASE_MODEL}")
print(f"reward = {ARMORM_MODEL}  (bf16, circular: RQ2 retired)")
print(f"out    = {OUT_DIR}")
print(f"steps={TOTAL_PPO_STEPS}  batch={BATCH_SIZE}  n_prompts={N_PROMPTS}")

**Cell 11**

## 5. Validate the config

Checks axis order and the ArmoRM settings the trainer relies on.

In [ ]:
# Cell 12
!python scripts/validate_tinyllama_helpsteer2_config.py

**Cell 13**

## 6. Create the theta_SFT snapshot

`run_ppo` builds the policy on top of `OUT_DIR/theta_sft/merged` and asserts it exists. This project uses the shortcut theta_SFT = base model (no separate SFT step), so we snapshot TinyLlama-Chat there once. `from_pretrained(...).save_pretrained(...)` is deterministic.

In [ ]:
# Cell 14
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

theta_sft = Path(OUT_DIR) / "theta_sft" / "merged"
if (theta_sft / "config.json").exists():
    print(f"theta_SFT already present at {theta_sft}")
else:
    theta_sft.mkdir(parents=True, exist_ok=True)
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    m.save_pretrained(str(theta_sft))
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(str(theta_sft))
    del m
    print(f"theta_SFT = base snapshot written to {theta_sft}")

**Cell 15**

## 7. Manual one-adapter-at-a-time PPO training

Each block trains exactly one axis by calling the tested trainer. ArmoRM (bf16) and the TinyLlama policy load together, so run **one axis at a time** on a single A100-40GB. The trainer automatically runs the golden-sample head-order check and proves batched == single scoring **before** the first PPO step; if either fails it stops on its own. After each run, check Cell 33 (section 9).

**`--armorm_load_in_4bit false` is now passed on every axis** — this is the fix for the golden-sample crash. Without it the trainer defaults to 4-bit (`CFG["armorm_load_in_4bit"]=True`) and the scorer aborts with `ArmoRM golden sample MISMATCH`. Setting the flag in a notebook Python cell has no effect: each `!python …` call is a fresh process that reads the CFG default, so the override must be on the command line.

The `--circular_armorm_acknowledged` flag is also required or the firewall refuses to run. All five axes share the same seeds and step count (Equal-N).

Note: the full head-discriminance gate from the old pipeline (does each ArmoRM head peak on its own axis on labeled text) is **not** in this thin notebook — a deliberate, reportable deviation to clear with Lingxiao before the binding run. The two hardest checks (head order via golden sample, batched-vs-single) do run automatically.

**Cell 16**

### 7.0 VRAM smoke-test (run once, throwaway)

Your manual golden-sample test loaded ArmoRM *alone*. A real PPO step also holds the TinyLlama policy, TRL's frozen reference copy, the generation cache (batch 64) and the PPO backward on the GPU at the same time. With bf16 ArmoRM (~16 GB) this is the tight case. This cell runs **3 PPO steps** of helpfulness against a throwaway directory (a symlink to the real θ_SFT so nothing is copied) and deletes it afterward, so it never looks like a finished axis. If it completes without OOM in a minute or two, the real axes are safe. If it hangs or is extremely slow, `device_map="auto"` has offloaded ArmoRM to CPU → not enough VRAM; drop `BATCH_SIZE` (keep it identical across all axes) or add an 8-bit load path.

In [ ]:
# Cell 17
import os, time, shutil, subprocess

probe_dir = "/tmp/vram_probe"
shutil.rmtree(probe_dir, ignore_errors=True)
os.makedirs(probe_dir, exist_ok=True)
# reuse the real theta_SFT snapshot (Cell 14) via symlink — no ~4 GB copy, OUT_DIR untouched
os.symlink(os.path.abspath(f"{OUT_DIR}/theta_sft"), f"{probe_dir}/theta_sft")

t0 = time.time()
try:
    subprocess.run([
        "python", "scripts/train_rs_ppo.py",
        "--phase", "ppo", "--axis", "helpfulness",
        "--reward_model", ARMORM_MODEL,
        "--circular_armorm_acknowledged",
        "--armorm_load_in_4bit", "false",
        "--out_dir", probe_dir,
        "--batch_size", str(BATCH_SIZE),
        "--total_ppo_steps", "3",
        "--n_prompts", "128",
    ], check=True)
    print(f"\nVRAM probe OK: 3 PPO steps in {time.time()-t0:.0f}s, no OOM. "
          f"bf16 ArmoRM + policy fit on this GPU — safe to run the real axes below.")
except subprocess.CalledProcessError:
    print("\nVRAM probe FAILED. If this was a CUDA OOM (or the run crawled), bf16 ArmoRM "
          "does not co-reside with the policy at this batch size. Options: lower BATCH_SIZE "
          "(same value for ALL five axes), or add an 8-bit ArmoRM load path in the scorer.")
    raise
finally:
    shutil.rmtree(probe_dir, ignore_errors=True)  # removes the symlink, not the real theta_SFT

**Cell 18**

### Helpfulness

In [ ]:
# Cell 19
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis helpfulness \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --armorm_load_in_4bit false \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

**Cell 20**

### Correctness

In [ ]:
# Cell 21
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis correctness \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --armorm_load_in_4bit false \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

**Cell 22**

### Coherence

In [ ]:
# Cell 23
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis coherence \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --armorm_load_in_4bit false \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

**Cell 24**

### Complexity

In [ ]:
# Cell 25
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis complexity \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --armorm_load_in_4bit false \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

**Cell 26**

### Verbosity

In [ ]:
# Cell 27
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis verbosity \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --armorm_load_in_4bit false \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

**Cell 28**

## 8. Check the running training

In [ ]:
# Cell 29
!pgrep -af "[t]rain_rs_ppo.py" || echo "No PPO training process is running."

**Cell 30**

Check the ArmoRM download cache if a run is still loading the reward model.

In [ ]:
# Cell 31
!du -sh /root/.cache/huggingface/hub/models--RLHFlow--ArmoRM-Llama3-8B-v0.1 2>/dev/null || echo "ArmoRM cache not created yet."

**Cell 32**

## 9. Inspect reward and plateau per axis

The trainer writes `ppo_log.json` per axis with the full reward log and a plateau interpretation. Two things to confirm for each finished axis:

- the reward curve **moved** and did **not** plateau — `plateau.interpretation` must say "still ascending"; a plateau breaks the short-horizon (upper-bound) premise;
- the **early** `mean_reward` is not far below the ArmoRM usable range (~0.66 raw) — if it is, 16–32-token generations may be too short for ArmoRM to score meaningfully (the open concern from the handoff; a deviation from RS output length would need to be justified to Lingxiao).

In [ ]:
# Cell 33
import json, glob
for path in sorted(glob.glob(f"{OUT_DIR}/ppo_*/ppo_log.json")):
    d = json.load(open(path))
    axis = d["axis"]; log = d["log"]
    early = sum(r["mean_reward"] for r in log[:10]) / max(1, len(log[:10]))
    late  = sum(r["mean_reward"] for r in log[-10:]) / max(1, len(log[-10:]))
    print(f"{axis:12}  early_reward={early:.4f}  late_reward={late:.4f}  "
          f"plateau={d['plateau'].get('reward_plateaued')}")
    print(f"             {d['plateau'].get('interpretation','')}")

**Cell 34**

## 10. Zip the adapters

Run after all five axes are finished. Contains each `ppo_<axis>/adapter/` plus its log and value head. This is the artifact NB05 consumes to compute R. Keep it out of Git.

In [ ]:
# Cell 35
!cd {OUT_DIR} && zip -r /content/rs_ppo_armorm_adapters.zip ppo_*/ -x "*/optimizer*" 
!ls -lh /content/rs_ppo_armorm_adapters.zip
print("Adapters are at:", *[f"{OUT_DIR}/ppo_{a}/adapter" for a in
    ["helpfulness","correctness","coherence","complexity","verbosity"]], sep="\n  ")

**Cell 36**

## 11. Git safety check

Adapters, snapshots, safetensors, value heads, and zips are generated artifacts. Keep them out of Git.

In [ ]:
# Cell 37
!git status